# ERA Translation Agent — Evaluation & Analysis

This notebook does two things to back up the project's claims with real numbers:

1. **Benchmark evaluation** — runs the pipeline over a small multilingual test set and
   measures LaBSE semantic similarity quality, retry rate, and how often the retry loop actually
   improves the result.
2. **Operational analysis** — reads the pipeline's own logged history (SQLite) to show
   quality trends and translation-memory hit rate over time.

**Requires:** `pip install -r ../requirements.txt` plus `matplotlib`, and a GPU is
recommended (CPU works but is slow). This notebook actually calls the models — it
was written and structured here but not executed in this environment (no GPU/network
access), so run it locally and paste the resulting numbers into the README.


In [ ]:
import sys, time
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

from src.era_translation_agent import TranslationPipeline

pipeline = TranslationPipeline()


## 1. Benchmark test set

Small hand-picked set across several language pairs. Swap this for FLORES-200 (`datasets.load_dataset("facebook/flores", ...)`) for a larger, standardized benchmark.

In [ ]:
test_set = [
    {"src_lang": "english", "tgt_lang": "spanish",  "text": "The quarterly report shows a 12% reduction in operating costs."},
    {"src_lang": "english", "tgt_lang": "french",   "text": "Please review the attached contract before Friday's meeting."},
    {"src_lang": "english", "tgt_lang": "german",   "text": "Our consultants identified significant savings in the procurement process."},
    {"src_lang": "english", "tgt_lang": "japanese", "text": "The client requested a revised proposal with updated pricing."},
    {"src_lang": "english", "tgt_lang": "polish",   "text": "All stakeholders must confirm the new terms by end of month."},
    {"src_lang": "english", "tgt_lang": "chinese",  "text": "The audit revealed discrepancies in the vendor invoices."},
]
len(test_set)


In [ ]:
results = []
for item in test_set:
    start = time.time()
    r = pipeline.translate(item["text"], item["src_lang"], item["tgt_lang"])
    elapsed = time.time() - start
    results.append({
        "src_lang": item["src_lang"],
        "tgt_lang": item["tgt_lang"],
        "source": item["text"],
        "translation": r["translation"],
        "quality": r["quality"],
        "attempts": r["attempts"],
        "cached": r["cached"],
        "seconds": round(elapsed, 2),
    })

df = pd.DataFrame(results)
df


### Did the retry loop help?

Compares quality on the first attempt vs. the final (possibly retried) result. Requires re-running with retries disabled for a fair A/B — the simplest proxy here is: how many rows needed `attempts > 1`, and what was their final quality?

In [ ]:
retried = df[df["attempts"] > 1]
print(f"{len(retried)}/{len(df)} translations needed a retry")
if len(retried):
    print(f"Average quality after retry: {retried['quality'].mean():.3f}")
retried


### Quality by language pair

In [ ]:
summary = df.groupby(["src_lang", "tgt_lang"])["quality"].mean().sort_values()
summary.plot(kind="barh", figsize=(6, 4), title="Avg LaBSE semantic similarity quality by language pair")
plt.axvline(0.85, color="red", linestyle="--", label="quality threshold")
plt.legend()
plt.tight_layout()
plt.show()


## 2. Operational analysis — the pipeline's own logged history

Every translation the pipeline runs gets logged to SQLite (`translation_system.db`). After you've used the app for a while, this section shows real usage trends rather than a one-off benchmark.

In [ ]:
stats = pipeline.db.get_statistics(num_supported_languages=29)
stats


In [ ]:
report = pipeline.db.generate_analytics_report(days=30)
daily = pd.DataFrame(report["daily_breakdown"])
print(f"Total translations (last 30 days): {report['total_translations']}")
print(f"Average quality: {report['avg_quality']:.3f}")

if len(daily):
    daily = daily.sort_values("date")
    daily.plot(x="date", y="avg_quality", marker="o", figsize=(7, 4), title="Average translation quality over time")
    plt.axhline(0.85, color="red", linestyle="--", label="quality threshold")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No logged history yet — run some translations through app.py first.")


## Summary

Fill this in after running locally, e.g.:

- Average LaBSE semantic similarity quality across the benchmark: **`_.___`**
- Retry rate: **`_/6`** translations needed a second attempt
- Best / worst performing language pair: **`___`** / **`___`**

Paste the headline number into the main `README.md` (e.g. *"Achieves an average LaBSE semantic similarity score of 0.XX across N language pairs"*) so the quality claim is backed by a number you actually measured, not just asserted.
